# 🔬 Model Training, Comparative Benchmarking & Clinical Uncertainty Quantification

**IIT Hyderabad — Semester Research Project**  
* **Supervised by**: Prof. Amit Acharyya & Dr. Pabitra Das  
* **Target Publication**: Computing in Cardiology (CinC) / IEEE EMBC / CHIL  

---

### Overview & Experimental Protocol
This notebook provides the interactive training and clinical evaluation suite for the ECG-to-Creatinine project:
1. **Tabular Demographic Controls**: Establishes performance floors (Population Mean, Linear Ridge, Extended Quadratic/Interaction Baseline).
2. **Deep Waveform Models**: Compares 1D-CNN Baseline (~1.1M params), 1D ResNet-34 (~7.3M params), and Pretrained ECG-FM (LoRA MHA adaptation).
3. **Publication Figures**: Generates Predicted vs. True scatter plots ($R^2$, Pearson $r$), Bland-Altman agreement plots, and KDIGO CKD confusion matrices.
4. **Clinical Uncertainty**: Computes Monte Carlo Dropout uncertainty intervals and reliability/calibration diagrams.


In [ ]:
import os
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader

# Clean plotting configuration
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8
warnings.filterwarnings('ignore')

# Resolve project root dynamically across local VS Code, cluster, or Google Colab
def _find_root() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd() / 'Mini Project',
        Path('/content/Mini Project'),
        Path('/content/Mini_Project'),
        Path('/content'),
    ]
    if 'ECG_PROJECT_ROOT' in os.environ:
        candidates.insert(0, Path(os.environ['ECG_PROJECT_ROOT']))
    for p in candidates:
        if (p / 'config' / 'pipeline_config.yaml').exists():
            return p.resolve()
    return Path.cwd().parent.resolve() if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config.paths import Paths, load_config
from datasets.ecg_dataset import ECGCreatinineDataset, build_dataloaders
from models import CNNBaseline1D, ResNet1D34, ECGFMLinearProbe, ECGFMLoRA, MockECGFMBackbone
from training.losses import build_loss
from training.trainer import Trainer, TrainerConfig
from evaluation.metrics import evaluate_all, ckd_screening_metrics
from evaluation.baseline_demographics import evaluate_demographic_baseline
from evaluation.uncertainty import mc_dropout_predict, calibration_plot_data, coverage_at_confidence
from evaluation.ckd_epi import egfr_ckdepi_2021, ckd_stage

cfg = load_config()
paths = Paths(cfg, root=PROJECT_ROOT)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Project root: {PROJECT_ROOT}')
print(f'Compute device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')


---
## 1. Load HDF5 Dataset & Build DataLoaders
Inspect train/val/test splits, verify tensor dimensions ($[B, 12, 5000]$), and confirm patient-level split isolation.


In [ ]:
print(f'Loading dataset from: {paths.hdf5_file}')
loaders = build_dataloaders(
    hdf5_path=paths.hdf5_file,
    batch_size=32,
    preload=False,
    num_workers=0,
)

sample_batch = next(iter(loaders['train']))
print(f'Train batches : {len(loaders["train"])}')
print(f'Val batches   : {len(loaders["val"])}')
print(f'Test batches  : {len(loaders["test"])}')
print(f'\nBatch waveform tensor : {sample_batch["waveform"].shape} ({sample_batch["waveform"].dtype})')
print(f'Batch label tensor    : {sample_batch["label"].shape} ({sample_batch["label"].dtype})')
print(f'Batch age tensor      : {sample_batch["age_years"].shape}')
print(f'Batch sex tensor      : {sample_batch["sex"].shape}')


---
## 2. Evaluate Control Demographic Baselines
Mandatory peer-review control benchmark. Evaluates whether waveform models provide statistically significant incremental prognostic value over demographic priors alone:
1. **Population Mean Baseline**: Predicts the training-set mean for every sample (absolute floor).
2. **Linear Demographic Baseline**: Closed-form Ridge regression on $[1, \text{age}, \text{sex}]$.
3. **Extended Demographic Baseline**: Closed-form Ridge on $[1, \text{age}, \text{sex}, \text{age}^2, \text{age} \times \text{sex}]$.


In [ ]:
demo_results = evaluate_demographic_baseline(paths.hdf5_file)

print('=' * 65)
print('  DEMOGRAPHIC CONTROL BASELINE RESULTS (TEST SPLIT)')
print('=' * 65)
for bl_name, metrics in demo_results.items():
    print(f'\n--- {bl_name.replace("_", " ").title()} ---')
    print(f'  Creatinine MAE : {metrics["mae"]:.4f} mg/dL')
    print(f'  Creatinine RMSE: {metrics["rmse"]:.4f} mg/dL')
    print(f'  Creatinine R²  : {metrics["r2"]:.4f}')
    print(f'  Pearson r      : {metrics["pearson_r"]:.4f}')
    if 'egfr_mae' in metrics:
        print(f'  eGFR MAE       : {metrics["egfr_mae"]:.2f} mL/min/1.73m²')
        print(f'  CKD Stage Acc  : {metrics["ckd_stage_acc"]:.2%}')


---
## 3. Model Architecture Instantiation & Parameter Counts
We compare three modeling tiers:
1. **1D-CNN Baseline**: 4 residual 1D convolutional blocks, adaptive average pooling, fully connected head (~1.1M parameters).
2. **1D ResNet-34**: 16 basic residual blocks adapted to 1D signals (~7.3M parameters).
3. **Pretrained Foundation Model (ECG-FM with LoRA)**: Transformer backbone with injected rank-8 LoRA adapters on Multihead Attention projections.


In [ ]:
models = {
    '1D-CNN Baseline': CNNBaseline1D(),
    '1D ResNet-34': ResNet1D34(),
    'ECG-FM Linear Probe': ECGFMLinearProbe(MockECGFMBackbone()),
    'ECG-FM LoRA Adapter': ECGFMLoRA(MockECGFMBackbone(), lora_rank=8, lora_alpha=16.0),
}

param_summary = []
for name, m in models.items():
    total_p = sum(p.numel() for p in m.parameters())
    train_p = sum(p.numel() for p in m.parameters() if p.requires_grad)
    param_summary.append({
        'Model': name,
        'Total Parameters': f'{total_p:,}',
        'Trainable Parameters': f'{train_p:,}',
        '% Trainable': f'{train_p / total_p * 100:.1f}%',
    })

pd.DataFrame(param_summary)


---
## 4. Interactive Model Training Loop
Train using:
- **Loss Function**: Huber Regression Loss ($\delta = 1.0$) for clinical robustness against extreme creatinine outliers.
- **Optimizer**: AdamW with Cosine Annealing learning rate schedule.
- **Gradient Clipping**: Maximum norm 1.0.
- **Early Stopping**: Patience of 5 epochs on validation loss.


In [ ]:
# Select model to train interactively
target_model_name = '1D ResNet-34'
model = models[target_model_name].to(device)

criterion = build_loss('huber', delta=1.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15, eta_min=1e-5)

ckpt_dir = PROJECT_ROOT / 'checkpoints' / 'interactive_run'
ckpt_dir.mkdir(parents=True, exist_ok=True)

t_cfg = TrainerConfig(
    epochs=15,
    learning_rate=1e-3,
    weight_decay=1e-4,
    grad_clip=1.0,
    patience=5,
    checkpoint_dir=ckpt_dir,
    device=str(device),
)

trainer = Trainer(
    model=model,
    train_loader=loaders['train'],
    val_loader=loaders['val'],
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    cfg=t_cfg,
)

print(f'Starting interactive training of {target_model_name} on {device}...')
history = trainer.fit()
print(f'Training complete. Best validation loss: {trainer.best_val_loss:.4f}')


---
## 5. Training Dynamics & Loss Curve Visualization
Plot the training and validation loss progression over epochs.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
epochs_range = range(1, len(history['train_loss']) + 1)
ax.plot(epochs_range, history['train_loss'], 'o-', color='#2980b9', label='Train Loss (Huber)', linewidth=1.8)
ax.plot(epochs_range, history['val_loss'], 's--', color='#e74c3c', label='Validation Loss (Huber)', linewidth=1.8)
ax.set_title(f'Training Dynamics: {target_model_name}', fontsize=12, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (Huber, delta=1.0)')
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.show()


---
## 6. Comprehensive Test Set Evaluation & Publication Plots
Generate publication-quality figures:
1. **Predicted vs. True Creatinine Scatter Plot** with $y = x$ line, $R^2$, and Pearson $r$.
2. **Bland-Altman Agreement Plot** showing systematic bias and 95% limits of agreement ($\pm 1.96 \text{ SD}$).
3. **KDIGO CKD Staging Confusion Matrix** (Stages G1 through G5).


In [ ]:
# Run inference on held-out test split
model.eval()
test_preds, test_trues, test_ages, test_sexes = [], [], [], []

with torch.no_grad():
    for batch in loaders['test']:
        x = batch['waveform'].to(device)
        pred = model(x).squeeze(-1).cpu().numpy()
        test_preds.extend(pred)
        test_trues.extend(batch['label'].numpy())
        test_ages.extend(batch['age_years'].numpy())
        test_sexes.extend(batch['sex'].numpy())

test_preds = np.clip(np.array(test_preds), 0.1, 25.0)
test_trues = np.array(test_trues)
test_ages = np.array(test_ages)
test_sexes = np.array(test_sexes)

# Compute full metrics
metrics = evaluate_all(test_trues, test_preds, age=test_ages, sex=test_sexes)

print('=' * 60)
print(f'  {target_model_name} TEST EVALUATION RESULTS')
print('=' * 60)
print(f'Creatinine MAE  : {metrics["mae"]:.4f} mg/dL')
print(f'Creatinine RMSE : {metrics["rmse"]:.4f} mg/dL')
print(f'Creatinine R²   : {metrics["r2"]:.4f}')
print(f'Pearson r       : {metrics["pearson_r"]:.4f}')
print(f'eGFR MAE        : {metrics["egfr_mae"]:.2f} mL/min/1.73m²')
print(f'CKD Stage Acc   : {metrics["ckd_stage_acc"]:.2%}')
print('=' * 60)

# Generate Publication Figures
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Predicted vs True Scatter Plot
axes[0].scatter(test_trues, test_preds, alpha=0.7, color='#2c3e50', edgecolor='w', s=45)
min_val = min(test_trues.min(), test_preds.min()) * 0.9
max_val = max(test_trues.max(), test_preds.max()) * 1.1
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5, label='y = x (Perfect)')
axes[0].set_title(f'Predicted vs. Ground Truth Creatinine\nR² = {metrics["r2"]:.3f}, r = {metrics["pearson_r"]:.3f}', fontsize=11, fontweight='bold')
axes[0].set_xlabel('True Creatinine (mg/dL)')
axes[0].set_ylabel('Predicted Creatinine (mg/dL)')
axes[0].set_xlim(min_val, max_val)
axes[0].set_ylim(min_val, max_val)
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].legend()

# 2. Bland-Altman Plot
diffs = test_preds - test_trues
means = (test_preds + test_trues) / 2.0
mean_diff = np.mean(diffs)
std_diff = np.std(diffs)

axes[1].scatter(means, diffs, alpha=0.7, color='#16a085', edgecolor='w', s=45)
axes[1].axhline(mean_diff, color='#2980b9', linestyle='-', linewidth=1.5, label=f'Mean Bias: {mean_diff:+.2f}')
axes[1].axhline(mean_diff + 1.96 * std_diff, color='#c0392b', linestyle='--', label=f'+1.96 SD: {mean_diff + 1.96*std_diff:+.2f}')
axes[1].axhline(mean_diff - 1.96 * std_diff, color='#c0392b', linestyle='--', label=f'-1.96 SD: {mean_diff - 1.96*std_diff:+.2f}')
axes[1].set_title('Bland-Altman Agreement Plot', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Mean of True and Predicted (mg/dL)')
axes[1].set_ylabel('Residual Difference (Pred - True)')
axes[1].grid(True, linestyle='--', alpha=0.5)
axes[1].legend(fontsize=8)

# 3. CKD Stage Confusion Matrix
egfr_t = egfr_ckdepi_2021(test_trues, test_ages, test_sexes)
egfr_p = egfr_ckdepi_2021(test_preds, test_ages, test_sexes)
stages_t = ckd_stage(egfr_t)
stages_p = ckd_stage(egfr_p)
labels_all = ['G1', 'G2', 'G3a', 'G3b', 'G4', 'G5']
present_labels = [l for l in labels_all if l in np.unique(np.concatenate([stages_t, stages_p]))]

# Manual confusion matrix calculation (clean-room, zero dependency)
cm = np.zeros((len(present_labels), len(present_labels)), dtype=int)
label_map = {l: i for i, l in enumerate(present_labels)}
for t_s, p_s in zip(stages_t, stages_p):
    if t_s in label_map and p_s in label_map:
        cm[label_map[t_s], label_map[p_s]] += 1

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=present_labels, yticklabels=present_labels, ax=axes[2])
axes[2].set_title(f'KDIGO CKD Staging Confusion Matrix\nAccuracy = {metrics["ckd_stage_acc"]:.1%}', fontsize=11, fontweight='bold')
axes[2].set_xlabel('Predicted Stage')
axes[2].set_ylabel('True Stage')

plt.tight_layout()
plt.show()


---
## 7. Monte Carlo Dropout Clinical Uncertainty Quantification
In acute clinical settings, point estimates alone are inadequate. We apply Monte Carlo Dropout (*Gal & Ghahramani, ICML 2016*):
- Keeps Dropout layers active at inference to draw $N=50$ stochastic samples from the approximate posterior predictive distribution.
- Yields epistemic uncertainty bounds ($10^{\text{th}}$ to $90^{\text{th}}$ percentiles).
- Evaluates calibration coverage against nominal confidence levels to plot reliability diagrams.


In [ ]:
# Run MC-Dropout inference on a batch of test samples
sample_batch = next(iter(loaders['test']))
n_uq_patients = min(16, len(sample_batch['label']))
x_mc = sample_batch['waveform'][:n_uq_patients].to(device)
y_mc = sample_batch['label'][:n_uq_patients].numpy()

uq_result = mc_dropout_predict(model, x_mc, n_samples=50, device=device)
means = uq_result['mean']
stds = uq_result['std']
p10 = uq_result['p10']
p90 = uq_result['p90']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Predictions with 90% Confidence Interval Error Bars
patient_indices = np.arange(len(means))
axes[0].errorbar(
    patient_indices,
    means,
    yerr=[means - p10, p90 - means],
    fmt='o',
    color='#2980b9',
    ecolor='#bdc3c7',
    elinewidth=2,
    capsize=4,
    label='MC-Dropout Pred (10th-90th %ile)',
)
axes[0].scatter(patient_indices, y_mc, color='#c0392b', marker='x', s=60, zorder=5, label='True Creatinine')
axes[0].set_title('Clinical Uncertainty: Predicted Creatinine with 90% CIs', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Test Patient Index')
axes[0].set_ylabel('Serum Creatinine (mg/dL)')
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].legend()

# 2. Calibration / Reliability Diagram
calib = calibration_plot_data(y_mc, uq_result['samples'])
axes[1].plot([0, 1], [0, 1], 'k--', label='Perfect Calibration')
axes[1].plot(calib['confidence_levels'], calib['empirical_coverages'], 's-', color='#8e44ad', linewidth=2, label='MC-Dropout Empirical')
axes[1].set_title('Uncertainty Calibration / Reliability Diagram', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Nominal Confidence Level')
axes[1].set_ylabel('Empirical Coverage Fraction')
axes[1].set_xlim(0.45, 1.0)
axes[1].set_ylim(0.45, 1.05)
axes[1].grid(True, linestyle='--', alpha=0.5)
axes[1].legend()

plt.tight_layout()
plt.show()

cov_90 = coverage_at_confidence(y_mc, means, stds, z=1.645)
print(f'Empirical Coverage at nominal 90% CI (z=1.645): {cov_90:.1%}')


---
## 8. Comparative Benchmark Summary Table
Collate all test results into a unified publication table comparing Tabular Demographic baselines directly against Deep Learning architectures:


In [ ]:
benchmark_data = [
    {
        'Model Type': 'Tabular Control',
        'Architecture': 'Population Mean',
        'Creatinine MAE (mg/dL)': f'{demo_results["mean_baseline"]["mae"]:.3f}',
        'RMSE (mg/dL)': f'{demo_results["mean_baseline"]["rmse"]:.3f}',
        'R²': f'{demo_results["mean_baseline"]["r2"]:.3f}',
        'Pearson r': f'{demo_results["mean_baseline"]["pearson_r"]:.3f}',
        'eGFR MAE (mL/min)': f'{demo_results["mean_baseline"].get("egfr_mae", np.nan):.2f}',
        'CKD Acc': f'{demo_results["mean_baseline"].get("ckd_stage_acc", np.nan):.1%}',
    },
    {
        'Model Type': 'Tabular Control',
        'Architecture': 'Linear Demographics (Age+Sex)',
        'Creatinine MAE (mg/dL)': f'{demo_results["demographic_baseline"]["mae"]:.3f}',
        'RMSE (mg/dL)': f'{demo_results["demographic_baseline"]["rmse"]:.3f}',
        'R²': f'{demo_results["demographic_baseline"]["r2"]:.3f}',
        'Pearson r': f'{demo_results["demographic_baseline"]["pearson_r"]:.3f}',
        'eGFR MAE (mL/min)': f'{demo_results["demographic_baseline"].get("egfr_mae", np.nan):.2f}',
        'CKD Acc': f'{demo_results["demographic_baseline"].get("ckd_stage_acc", np.nan):.1%}',
    },
    {
        'Model Type': 'Tabular Control',
        'Architecture': 'Extended Demographics (Age²+Age*Sex)',
        'Creatinine MAE (mg/dL)': f'{demo_results["extended_demographic_baseline"]["mae"]:.3f}',
        'RMSE (mg/dL)': f'{demo_results["extended_demographic_baseline"]["rmse"]:.3f}',
        'R²': f'{demo_results["extended_demographic_baseline"]["r2"]:.3f}',
        'Pearson r': f'{demo_results["extended_demographic_baseline"]["pearson_r"]:.3f}',
        'eGFR MAE (mL/min)': f'{demo_results["extended_demographic_baseline"].get("egfr_mae", np.nan):.2f}',
        'CKD Acc': f'{demo_results["extended_demographic_baseline"].get("ckd_stage_acc", np.nan):.1%}',
    },
    {
        'Model Type': 'Deep Learning (ECG)',
        'Architecture': f'{target_model_name}',
        'Creatinine MAE (mg/dL)': f'{metrics["mae"]:.3f}',
        'RMSE (mg/dL)': f'{metrics["rmse"]:.3f}',
        'R²': f'{metrics["r2"]:.3f}',
        'Pearson r': f'{metrics["pearson_r"]:.3f}',
        'eGFR MAE (mL/min)': f'{metrics["egfr_mae"]:.2f}',
        'CKD Acc': f'{metrics["ckd_stage_acc"]:.1%}',
    },
]

df_benchmark = pd.DataFrame(benchmark_data)
print('=' * 75)
print('  FINAL COMPARATIVE BENCHMARK TABLE')
print('=' * 75)
df_benchmark
